# Why do people smoke ?

Every year there are numerous death due to smoking. On an average -

1. Over 7 million deaths annually are due to direct tobacco use. <br>

2. Around 1.3 million deaths are caused by secondhand smoke exposure. <br>

3. Smoking-related deaths are expected to rise to 10 million annually by 2030 if current trends continue.

We often come across news propogating the dangers of smoking and why one shouldn't indulge in this habit. However Public measures aren't often reaching the people to bring about an actual behavioural change.<br>

The aim of this project was to find the factors contributing to this fatal habit to understand the reasons why one could be persuaded to hold on to this habit inspite of the health concerns. <br>

We wanted to also categorize these contributing factors of smoking to various categories of the Health Belief Model (HBM) to understand the root cause of people normalizing the act of smoking. 



## Health Belief Model

The Health Belief Model (HBM) is a psychological framework developed in the 1950s to explain and predict individual health behaviors, particularly regarding preventive health actions. It is one of the most widely used theories in health psychology, public health, and behavioral science.

This model attributes a persons health related behaviour to 6 key factors - <br>

1. Perceived Susceptibility <br>

“How likely am I to get this condition?”<br>
A person’s belief about the chances of getting a disease or health issue.<br>
Example: "Because I have a family history of smoking-related illness, I think I’m at risk."<br>

2. Perceived Severity<br>

“If I get it, how bad will it be?”<br>
Belief about the seriousness of the condition and its potential consequences.<br>
Example: "Lung cancer is deadly and could seriously affect my life." <br>

3. Perceived Benefits <br>

“Will taking action help?”<br>
Belief that a recommended action will reduce the risk or severity of the condition.<br>
Example: "If I quit smoking, I’ll reduce my risk of disease."<br>

4. Perceived Barriers <br>

“What’s stopping me?”<br>
Beliefs about the costs or obstacles to performing the recommended action.<br>
Example: "I’m afraid I’ll gain weight or be stressed if I quit smoking."<br>

5. Cues to Action<br>

“What triggers the behavior?”<br>
Internal or external triggers that prompt behavior change.<br>
Examples: Media campaigns, illness of a loved one, doctor's advice.<br>

6. Self Efficacy<br>

"How motivated am I to do what's right?"<br>
Internal beliefs that promote making healthy choices.<br>
Examples: Staying active because its important to include movement in daily schedule


## Understand Behavioral Risk Factor Surveillance System (BRFSS) data

The data was imported from Kaggle and contains Survey data for years 2011-2015.

It has (2380047, 645) records. <br>

However to start our analysis, we started by performing basic data wrangling and eda steps and carried out visualtization and data summarization to answer few questions mentioned below




## Questions considered

1. Can we predict smokers from non-smokers using this dataset. If yes - <br>


    a. What is our target variable ?<br>
    We evaluate various smoking variables throughout our Data Wrangling and EDA procedure and choose _RFSMOK3 as the predictor variable.<br>

2. Are there any variables close to target variable that will influence our prediction model <br>

3. Are there duplicate variables in the feature set, should we drop some of them ?<br>

4. Are all features converted to correct data types <br>
    a. Categorical <br>
    b. Ordinal

5. Of the existing categorical values, can you reduce some into fewer buckets<br>

6. What is the distribution of Numerical features<br>

7. Can you show relationship of individual features to the target variable <br>

8. Is there any pattern in the missingness of features ? <br>
    a. Should we impute with mean or median ? <br>
    b. Should we drop the rows 


***********************************************************************************************



## DATA WRANGLING -

1. Import BRFSS data. <br>
    1. Package - os, glob, kagglehub, pandas, re
    2. Some data Characteristics are - -<br>
        1. Large data - ~2.5 million records, 645 columns
        2. Survey data (can expect missing columns) - cue to think about feature engineering techniques + ML algorithms that can handle large number of missing values
2. Ydata Profiling (to help with visualizing missingness of columns and distributions) at a glance - <br>
    1. Package - from ydata_profiling import ProfileReport -> `profileReport = ProfileReport(remove_missing_cols_75, minimal=True)`; 
    2. Some benefits of using it -
        1. view Missingness in features (1. Drop features with >=75% missing data)
            1. We dropped variables with >75% missing values after confirming they were not critical to business goals and not strongly correlated with target.
        2. view Skewed features
        3. view Unsupported features (possibly because all values are missing)
        4. Perform Data Quality Checks and EDA
3. For remaining 87 features, we perform some tasks like - <br>
    1. drop Record Identification features that don't help in predicting smoking
    2. Drop features that appear to be duplicate -> `df.drop(['colname'], axis=1)`
    3. Fix the data type of features 
        1. Convert Numeric features to Categorical (Nomial / Ordinal based on context)
            1. Ordinal Dtype - 
            `from pandas.api.types import CategoricalDtype`
            `def convertToOrdinalCategoricalFeature(df, columnName, order_list):`
                `df[columnName] = df[columnName].fillna(7)`
                `df[columnName] = df[columnName].astype('int')`

    # Define the order of the categories
    ordered_cat = CategoricalDtype(categories=order_list, ordered=True)
    # Apply the ordered category using .astype()
    df[columnName] = df[columnName].astype(ordered_cat)
    return df[columnName]
    4. Unit Normalization + Feature scaling/Binning (MENTHLH; PHYSHLTH) 
        1. Had a lot of zeros i.e zero-skewed
        2. step-wise relationship with Target Variable
 
Some things to think about - <br>
1. Pick the right threshold to discard missing values. 
    1. Survey data usually contains lots of missing data. We evaluated using ydata-profiling that most of the variables with > 75% of missing values didn't contribute towards the target varialbe which is variable capturing smoking habits. Hence we decide to drop it.

    | Missing %  | Recommendation                                            |
    | ---------- | --------------------------------------------------------- |
    | **> 80%**  | Usually drop (too little data to impute reliably)         |
    | **50–80%** | Drop **only if** not critical; else impute or engineer    |
    | **< 50%**  | Prefer imputation (mean/median, KNN, model-based)         |
    | **< 20%**  | Safe to keep; impute or leave if model can handle missing |



2. Identify how remaining missingness can be handled -<br>
    1. MCAR (Missing Completly At Random)
    2. MAR (Missing At Random)
    3. MNAR (Missing Not At Random)
3. Distribution of features to guide selection of appropriate Feature Engineering + ML algorithms
4. Think about the task at hand, let it guide your choice of ML model and feature transformation.
    1. ML Model choice -
        1. Catboost (Natively handles missing values + categorical data + Handles Skewed Data)
        2. LightGBM (Faster + handles missing values)
        3. XGBoost
        4. Random Forest
        5. Logistic Regression (Needs scaling + missing value imputation)

    #### Model Sensitivity
    | Model Type                                      | Works Well With          | Recommendation                |
    | ----------------------------------------------- | ------------------------ | ----------------------------- |
    | **Linear models (Logistic, Linear Regression)** | Continuous, normal-like  | ➤ **Log Transform**           |
    | **Tree-based models (XGBoost, CatBoost, RF)**   | Handle skew well         | ➤ **May not need transform**  |
    | **Naïve Bayes / KNN / SVM**                     | Scaled + normalized data | ➤ **Log Transform preferred** |
    | **Rule-based / interpretable models**           | Categories               | ➤ **Binning**                 |

    2. Feature Engineering + Data Wrangling Choice 
        1. Converting features having real (exhaustive numbers) representing various code in the CDC workbook, were converted to category datatype
        2. Missing values were kept as it is as missing values
        3. Decisions to apply Log-Transformation or Binning were done after checking -
            1. Binning -> if there is a step-wise relationship between feature and target variable <br>
                       -> convert to categories (ordinal bins)<br>
                       -> if there are many outliers <br>
                       -> If target mean changes step-wise across bins → binning is helpful <br>
                       -> `pd.qcut(df['income'], q=5, labels=False)  # quantile bins`
            2. Log-Transformation -> if the values are +ve and skewed <br>
                                  -> preserve numeric continuity <br>
                                  -> If target mean changes smoothly → log transformation is better.
            
| Skewness Value                | Suggestion                              |
| ----------------------------- | --------------------------------------- |
| 0 to ±0.5                     | No correction needed                    |
| > 0.5                         | Right-skewed → consider **log**         |
| < -0.5                        | Left-skewed → consider **square / exp** |
| Extremely skewed (many zeros) | consider **binning or log1p**           |

#### Summary Decision Table
| Scenario                                         | Recommended Approach            |
| ------------------------------------------------ | ------------------------------- |
| Continuous numeric, right-skewed                 | **Log / Yeo-Johnson transform** |
| Feature has many zeros                           | **log1p** or **binning**        |
| Relationship with target is step-like            | **Binning**                     |
| Need interpretability (e.g. reports, SHAP plots) | **Binning**                     |
| Need smooth feature for linear model             | **Log**                         |
| Using tree-based models                          | Often **no transform needed**   |

### General DATA WRANGLING Rules 

Goal: Ensure data quality, handle missing values, and create a usable dataset.

#### Step 1.1: Load & Inspect

1. `df.shape`, `df.info()`, `df.describe()`
2. Identify datatypes: numeric, categorical, ordinal, binary, date, text
3. Identify target distribution (smoking_status.value_counts())

#### Step 1.2: Missingness Audit

1. Compute % missing per column:
`missing = df.isnull().mean().sort_values(ascending=False)`
2. Visualize with heatmap or bar plot

    Classify:

    1. Completely missing (MCAR)
    2. Systematic missing (e.g., income missing more in young)
    3. Structural missing (e.g., irrelevant by logic)

    Action:

    1. Drop columns with >60–70% missing if non-informative or imputation unreliable
    2. Keep others; plan imputation

#### Step 1.3: Data Type Conversion

1. Convert categorical codes to category
2. Convert dates to datetime
3. Ensure numeric types are correct

#### Step 1.4: Outlier Check

1. Use IQR or modified z-score for numeric features
2. For large survey data, prefer capping (winsorization) over removal

#### Step 1.5: Data Consistency

1. Ensure all values in valid ranges (e.g. age > 0, < 120)
2. Standardize category labels (“Male”, “M”, “male” → “Male”)



## EXPLORATORY DATA ANALYSIS -

#### 1. HBM mapping for BRFSS features

 The features in hbm_features were selected based on mapping BRFSS survey variables to the constructs of the Health Belief Model (HBM).
 The process involved:

1. Reviewing the HBM constructs: Perceived Susceptibility, Perceived Severity, Perceived Benefits, Perceived Barriers, Cues to Action, and Self-Efficacy.
2. Reading BRFSS codebook and variable descriptions to understand what each variable measures.
3. Assigning variables to HBM constructs based on their meaning:
    - If a variable measures risk factors or history of disease, it maps to Susceptibility.
    - If it measures health status or limitations, it maps to Severity.
    - If it measures access to care, preventive actions, or insurance, it maps to Benefits.
    - If it measures obstacles like cost or SES, it maps to Barriers.
    - If it measures triggers or prompts for action (e.g., doctor advice, symptoms), it maps to Cues to Action.
    - If it measures confidence, ability, or successful health behaviors, it maps to Self-Efficacy.
    - If it has information about height, weight, age it maps to Other variables to capture Demographical Information.
4. Only variables present in the cleaned BRFSS dataframe (df.columns) were included.
5. Variables were grouped under each HBM construct for interpretability and to guide further analysis.

#### 2. Run following checks for each construct -

1. Check imbalance between features
2. Plot relationship between feature and target variable `_RFSMOK3`
3. Convert duplicate categorical codes (7,9 to 7), establish consistency in dataset. Check for the % of (7, 9 ) codes to evaluate missingness of the feature, if it is > 5%, check if there is any pattern suggestive by the missingness itself. Do it by introducing flags

#### 3. Outlier analysis for features to make sure that the complete data is legit

| Strategy             | When to Use                           | Example                                 |
| -------------------- | ------------------------------------- | --------------------------------------- |
| **Remove**           | Clear data errors or extreme invalids | Income = -100                           |
| **Cap (Winsorize)**  | Keep distribution but limit influence | Clip to Q1–1.5*IQR, Q3+1.5*IQR          |
| **Transform**        | Skewed features                       | Log / Yeo-Johnson                       |
| **Model Robustly**   | Tree-based models                     | RandomForest, CatBoost handle them well |
| **Impute / Replace** | If due to missing placeholder         | Replace 999 → NaN                       |

#### 3. Outlier Summary Table

| Method               | Good For        | Notes          |
| -------------------- | --------------- | -------------- |
| **IQR**              | Simple 1D       | Non-parametric |
| **Z-score**          | Normal data     | Parametric     |
| **Modified Z-score** | Skewed data     | Robust         |
| **Isolation Forest** | High-dim data   | Multivariate   |
| **LOF**              | Local anomalies | Density-based  |
| **Boxplot**          | Visual EDA      | Easy & fast    |

### General EDA steps

Goal: Understand structure, relationships, distribution, missingness patterns.

#### Step 2.1: Univariate Analysis

1. Numerical: Histograms, skewness
2. Categorical: Value counts
3. Target: Class balance (e.g. smokers %)

#### Step 2.2: Bivariate Analysis

1. Compare features vs smoking_status
2. Categorical: Chi-square / barplots
3. Numeric: Boxplots, t-tests

Identify key predictors (e.g., Age, Education, Income, Region)

#### Step 2.3: Correlation & Collinearity

1. For numeric → df.corr()
2. For categorical → Cramér’s V
3. Drop redundant features (highly correlated)

#### Step 2.4: Missingness Patterns

1. Use missingno.matrix(df) or check if missingness correlates with smoking
2. If yes, missingness might be informative (→ create a “missing flag” feature)

#### MISSINGNESS Discussed

##### 1. Visualize Missingness

1. Use `missingno` to quickly visualize missing data patterns:

        import missingno as msno

        msno.matrix(df)          # visualize missing cells
        msno.heatmap(df)         # show correlation between missingness

2. What to look for:
    1. Columns with similar missing patterns
    2. Blocks of missing values across certain subsets (e.g., all health-related questions missing for a group)

3. If two variables are often missing together, they might share a cause (e.g. optional section skipped by some respondents).

##### 2. Quantify Missingness

1. Get missingness ratio per column:

    `missing_summary = df.isnull().mean().sort_values(ascending=False)`

2. Example output:

        income           0.45
        alcohol_use      0.30
        exercise_freq    0.12
        age              0.00

This shows 45% of rows are missing income → too large to ignore without understanding.

##### 3. Check Correlation of Missingness with Target

1. We now check: Is the missingness informative?

To do that:
1. Create a missing indicator variable for each feature:
`df['income_missing'] = df['income'].isnull().astype(int)`

2. Now:
`df.groupby('income_missing')['smoking_status'].mean()`

3. Example result:

income_missing	mean(smoking_status=1)
0 (not missing)	0.32 (32% smokers)
1 (missing)	0.51 (51% smokers)

    3.1. Interpretation:
    People who did not report income have higher smoking rates.
    So the missingness itself is predictive.
You should keep income_missing as a binary feature in your model.

##### 4. Statistical Check

1. You can also use a Chi-square test for categorical target or t-test for continuous target to test significance:

`from scipy.stats import chi2_contingency` <br>
`contingency = pd.crosstab(df['income_missing'], df['smoking_status'])`<br>
`chi2, p, dof, expected = chi2_contingency(contingency)`<br>
`print("p-value:", p)` <br>


If p < 0.05 → missingness is significantly related to the target

##### 5. Repeat for Other Variables

1. Do this for other columns:

`for col in df.columns:`<br>
    `if df[col].isnull().any():`<br>
        `df[col + '_missing'] = df[col].isnull().astype(int)`

2. Then check which _missing features show strong relationship with smoking_status.

##### 6. Use Missing Flags in Modeling

Add those _missing features to your final dataset.
They act as informative binary predictors (0 = known, 1 = missing).

##### 6.  Works great with:

Logistic Regression (linear models)

Tree-based models (Random Forest, XGBoost, CatBoost)

In tree models, these missing flags help split the data more effectively.



## 3. Feature Engineering

Goal: Transform raw data into model-ready features.

##### Step 3.1: Imputation
Handle missingness differently by type:

Type	Example	Strategy
1. Numeric	age, income	Median or model-based imputation (KNNImputer)
2. Categorical	education, marital_status	Mode or “Unknown”
3. Ordinal	education_level	Mode or median rank
4. Indicator	Create _missing_flag feature for columns with >10% missing	

💡 For tree-based models, missing values can be left as-is (XGBoost, CatBoost handle missing natively).
💡 For linear/logistic models, impute explicitly.

##### Step 3.2: Encoding
Variable Type	Method	Notes
1. Binary	Map to 0/1	e.g. sex: male=0, female=1
2. Nominal (unordered categorical)	One-hot encoding	Use `pd.get_dummies()`
3. Ordinal (ranked categories)	Label encoding by order	e.g. “Low”=1, “Medium”=2
4. High-cardinality categorical	Target encoding / frequency encoding	Especially for ID-like features

💡 For CatBoost, you can feed categorical columns as-is (no encoding needed).

binary_cols = ['sex', 'married']
nominal_cols = ['region', 'occupation_type']
ordinal_cols = ['education_level']
target = 'smoking_status'

###### 1.  Binary
`for col in binary_cols:`<br>
    `df[col] = df[col].map({'No': 0, 'Yes': 1})  # or {'Male': 0, 'Female': 1}`

###### 2.  Nominal
`df = pd.get_dummies(df, columns=nominal_cols, drop_first=True)`

###### 3.  Ordinal
`edu_map = {'Primary': 1, 'High School': 2, 'College': 3, 'Graduate': 4}`
`df['education_level'] = df['education_level'].map(edu_map)`

###### 4.  High-cardinality (optional)
`from category_encoders import TargetEncoder`
`encoder = TargetEncoder(cols=['occupation'])`
`df['occupation'] = encoder.fit_transform(df['occupation'], df[target])`

Target encoding -
1. Captures predictive signal
2. Must do within cross-validation to prevent data leakage!

Frequency encoding -
1. Simple, model-agnostic
2. Doesn’t capture target relationship, only popularity

| Model Type                          | Encoding Recommendation                      |
| ----------------------------------- | -------------------------------------------- |
| **Linear / Logistic Regression**    | One-hot for nominal, label for ordinal       |
| **Tree-based (XGBoost, CatBoost)**  | Label encoding or raw categorical (CatBoost) |
| **High-cardinality + Linear model** | Target encoding                              |
| **Binary columns**                  | Map to 0/1                                   |


##### Step 3.3: Scaling (if needed)
Depends on model:
1. Tree-based models: No scaling needed
2. Logistic Regression / SVM / KNN: StandardScaler or MinMaxScaler<br>
                                  : If skewed → apply log transform first

##### Step 3.4: Binning (optional)
1. For skewed continuous vars: create bins (age groups)
For income: quantile bins `(pd.qcut)`
Helps interpretability for Logistic Regression

##### Step 3.5: Feature Creation
1. Combine features (e.g., BMI = weight / height²)
2. Ratios (e.g., income per household)
3. Missing indicators
4. Interaction terms if relevant


## KEY NOTES

### 1. Understand why people engage or do not engage in preventive health behaviour
### 2. Can we find few consistent beliefs in peopled belonging to similar age groups who engage in this activity or is it random ?

# Assumptions -




